# 01. Data Inspection — DiaGraMo

## Mục tiêu

Kiểm tra tính toàn vẹn và cấu trúc dữ liệu DiaGraMo
để phục vụ tái nghiệm bài báo:

*Towards Objective Dysgraphia Detection:
A Multi-Branch Deep Learning Approach
for Online Handwriting Analysis.*

Các nội dung cần kiểm tra:

1. Số lượng participants và recordings.
2. Cấu trúc 7 tín hiệu handwriting.
3. Tính nhất quán về số lượng mẫu.
4. Missing values và các bất thường trong tín hiệu.
5. Tính đầy đủ của 14 tác vụ mỗi participant.
6. Xác định tập 258 participants theo bài báo.

**Nguyên tắc:** Chỉ đọc và kiểm tra dữ liệu,
không chỉnh sửa các file JSON gốc.

In [1]:
from pathlib import Path
from collections import Counter
import json
import numpy as np
import pandas as pd

# Locate the original clean dataset
cwd = Path.cwd().resolve()

candidates = [
    cwd / "data" / "clean_data",
    cwd.parent / "data" / "clean_data",
    cwd / "clean_data",
]

DATA_DIR = next(
    (p for p in candidates if p.is_dir()),
    None
)

if DATA_DIR is None:
    raise FileNotFoundError(
        "Cannot find data/clean_data. "
        f"Current directory: {cwd}"
    )

json_files = sorted(DATA_DIR.glob("*_merged.json"))

print("=" * 50)
print("DIAGRAMO DATA INVENTORY")
print("=" * 50)
print(f"Data directory : {DATA_DIR}")
print(f"Total JSON files: {len(json_files)}")
print(f"First file      : {json_files[0].name if json_files else 'N/A'}")
print(f"Last file       : {json_files[-1].name if json_files else 'N/A'}")

assert len(json_files) == 294, (
    f"Expected 294 JSON files, found {len(json_files)}"
)


DIAGRAMO DATA INVENTORY
Data directory : E:\DiaGraMo-project\data\clean_data
Total JSON files: 294
First file      : BR09401_merged.json
Last file       : VV10404_merged.json



## 1. Participant and Task Inventory

Theo Section II.B của bài báo, tác giả lựa chọn 258 participants
có đầy đủ 14 tác vụ handwriting.

Trong bước này, chúng ta sẽ:

1. Đọc lần lượt 294 file JSON.
2. Xác định participant ID từ tên file.
3. Đếm số recording và số task ID khác nhau.
4. Kiểm tra sự hiện diện của cognitive test data.
5. Thống kê những participants có đủ hoặc thiếu tác vụ.

**Lưu ý:** Chưa loại bỏ participants. Kết quả chỉ được
sử dụng để đánh giá tính đầy đủ của dữ liệu.

In [2]:
import re
import json
from collections import Counter

records = []

for file_path in json_files:
    participant_id = file_path.stem.removesuffix("_merged")

    # Load one participant at a time
    with open(file_path, "r", encoding="utf-8") as f:
        participant = json.load(f)

    # Identify handwriting tasks
    task_ids = []

    for key in participant.keys():
        match = re.search(r"_TSK(\d+)\(", key)

        if match:
            task_ids.append(int(match.group(1)))

    has_cognitive = any(
        "cognitive_test_data" in key
        for key in participant.keys()
    )

    records.append({
        "participant_id": participant_id,
        "n_recordings": len(task_ids),
        "n_unique_tasks": len(set(task_ids)),
        "has_cognitive": has_cognitive,
        "complete_14_tasks": (
            len(task_ids) == 14
            and len(set(task_ids)) == 14
        )
    })

inventory_df = pd.DataFrame(records)

print("=" * 50)
print("PARTICIPANT & TASK INVENTORY")
print("=" * 50)

print("Total participants:", len(inventory_df))
print("Total recordings:", inventory_df["n_recordings"].sum())

print(
    "Participants with 14 tasks:",
    inventory_df["complete_14_tasks"].sum()
)

print(
    "Participants with incomplete tasks:",
    (~inventory_df["complete_14_tasks"]).sum()
)

print("\nTask count distribution:")
print(
    inventory_df["n_unique_tasks"]
    .value_counts()
    .sort_index()
)

print("\nParticipants with missing tasks (first 15):")
display(
    inventory_df.loc[
        ~inventory_df["complete_14_tasks"]
    ].head(15)
)


PARTICIPANT & TASK INVENTORY
Total participants: 294
Total recordings: 4000
Participants with 14 tasks: 266
Participants with incomplete tasks: 28

Task count distribution:
n_unique_tasks
0       6
11      1
12      8
13     13
14    266
Name: count, dtype: int64

Participants with missing tasks (first 15):


,participant_id,n_recordings,n_unique_tasks,has_cognitive,complete_14_tasks
0,BR09401,0,0,True,False
1,BR09402,0,0,True,False
2,BR09403,0,0,True,False
3,BR09404,0,0,True,False
6,BR10303,13,13,True,False
10,BR10401,12,12,True,False
11,BR10402,13,13,True,False
43,BR3405,13,13,True,False
90,BR7309,13,13,True,False
99,BR7318,13,13,True,False


## Kiểm tra hoàn chỉnh từ pipeline tái lập

294 tệp nguồn không đồng nghĩa 294 người đủ điều kiện. Phân biệt đủ 14 tác vụ, có nhãn và các cờ loại.

In [3]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
ROOT = Path.cwd() if (Path.cwd() / 'data').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / 'src'))
print('Project:', ROOT)


Project: E:\DiaGraMo-project


In [4]:
print(json.dumps(json.loads((ROOT/'results/audit/summary.json').read_text()),indent=2))
audit=pd.read_csv(ROOT/'results/audit/participants.csv')
audit.groupby(['task_count','group'],dropna=False).size().to_frame('participants')

{
  "files": 294,
  "recordings": 4000,
  "complete_14": 266,
  "included": 258,
  "class_counts": {
    "0": 115,
    "1": 143
  },
  "nonfinite_samples": 0,
  "nonincreasing_timestamps": 23455
}


participants
task_count group              
0          0.0               2
           1.0               1
           NaN               3
11         1.0               1
12         0.0               1
           1.0               6
           NaN               1
13         0.0               4
           1.0               7
           NaN               2
14         0.0             115
           1.0             143
           NaN               8

In [5]:
r=pd.read_csv(ROOT/'results/audit/recordings.csv')
print('Nonfinite samples:',r.nonfinite.sum())
print('Timestamp steps not increasing:',r.nonincreasing_timestamps.sum())
print('Metadata sample count mismatches:',(r.n_samples!=r.meta_samples).sum())
audit[audit.included & (audit.group!=audit.diagnosis_dysgraphia)][['participant_id','group','diagnosis_dysgraphia']]

Nonfinite samples: 0
Timestamp steps not increasing: 23455
Metadata sample count mismatches: 7


,participant_id,group,diagnosis_dysgraphia
7,BR10304,0.0,1.0
9,BR10306,1.0,0.0
113,BR7408,1.0,0.0
117,BR7412,1.0,0.0
159,HK03304,1.0,0.0
